In [ ]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from google.colab import drive

# confgurar el Drive
drive.mount('/content/drive')
carpeta_train = "/content/drive/MyDrive/dataset_colon/train"
carpeta_test = "/content/drive/MyDrive/dataset_colon/test"

# cargamos los Datasets
batch_size = 32
img_size = (240, 240)
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    carpeta_train, image_size=img_size, batch_size=batch_size, label_mode='int', shuffle=True)
test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    carpeta_test, image_size=img_size, batch_size=batch_size, label_mode='int', shuffle=False)
class_names = train_dataset.class_names

# configuramos los pesos de Clase
y_train = np.concatenate([y for x, y in train_dataset], axis=0)
pesos = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(pesos))

# optimizaciÃ³n
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.4),
    layers.RandomZoom(0.3),
    layers.RandomContrast(0.2)
])

# Modelo Base: EfficientNetV2B1 (MÃ¡s profundo que el B0)
base_model = tf.keras.applications.EfficientNetV2B1(
    input_shape=(240, 240, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # FASE 1: Congelado

# 7. ConstrucciÃ³n de la red con funcion de activaciÃ³n Mish
inputs = tf.keras.Input(shape=(240, 240, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)
# Usamos 'mish' en lugar de 'relu' (mejor flujo de gradientes)
x = layers.Dense(512, activation='mish', kernel_regularizer=regularizers.l2(0.01))(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='mish', kernel_regularizer=regularizers.l2(0.01))(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs)

# FASE 1: Entrenamiento de la Cabeza

print("\n--- FASE 1: Calentamiento ---")
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=12,
    class_weight=class_weights_dict
)

# FASE 2: FINE-TUNING EXTREMO (Truco de Batch Normalization)

print("\n--- FASE 2: Fine-Tuning Extremo ---")
# Descongelamos la base entera...
base_model.trainable = True

# volvemos a congelar explÃ­citamente las capas de Batch Normalization.
# Esto evita que la red destruya lo aprendido al recalcular estadÃ­sticas.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Tasa ultra baja
    metrics=["accuracy"]
)

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=35,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights_dict
)

# 8. EvaluaciÃ³n Final
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\n>> ACCURACY FINAL: {test_accuracy * 100:.2f}% <<\n")

y_pred_valor = model.predict(test_dataset)
y_pred = np.argmax(y_pred_valor, axis=1)
y_true = np.concatenate([y for x, y in test_dataset], axis=0)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

plt.figure(figsize=(10, 8))
disp.plot(cmap='Blues', xticks_rotation=45, values_format='d')
plt.title("Matriz de ConfusiÃ³n", fontsize=14, fontweight='bold')
plt.xlabel("PredicciÃ³n", fontsize=12)
plt.ylabel("Real", fontsize=12)
plt.tight_layout()
plt.savefig('/content/matriz_confusion_final.png', dpi=300)
plt.show()